**© Copyright AIDENTIFY. All rights reserved.**

# Part 4 | Session 26: RAG 파이프라인 - ChromaDB & 시맨틱 서치

## 📋 학습 목표

1️⃣ RAG (Retrieval-Augmented Generation)의 핵심 개념을 이해한다  
2️⃣ 문서를 로딩하고 적절한 크기로 청킹하는 방법을 배운다  
3️⃣ 임베딩을 생성하고 ChromaDB에 저장하는 실습을 한다  
4️⃣ 시맨틱 서치로 관련 문서를 검색한다  

---

### 🖥️ 실습 환경
- **GPU**: 불필요 (CPU로 충분)
- **필수 패키지**: chromadb, sentence-transformers, langchain

## 1️⃣ RAG란 무엇인가?

**RAG (Retrieval-Augmented Generation)**은 LLM의 한계를 보완하기 위해 **외부 지식을 검색하여 답변에 활용**하는 기술입니다.

### 🔑 LLM의 한계
- 🔹 학습 데이터 이후의 정보를 모름 (지식 컷오프)
- 🔹 회사 내부 문서, 개인 데이터를 모름
- 🔹 할루시네이션 (사실과 다른 내용 생성)

### 🏗️ RAG 파이프라인 구조

```
┌─────────────┐    ┌──────────────┐    ┌──────────────┐
│  1. 문서 로딩  │ → │  2. 청킹/분할  │ → │  3. 임베딩 생성 │
└─────────────┘    └──────────────┘    └──────────────┘
                                              │
                                              ▼
┌─────────────┐    ┌──────────────┐    ┌──────────────┐
│  6. LLM 응답  │ ← │  5. 프롬프트   │ ← │  4. 벡터DB 저장 │
└─────────────┘    └──────────────┘    └──────────────┘
```

In [ ]:
# 📦 필수 패키지 확인
import importlib

packages = [
    "chromadb",
    "sentence_transformers",
    "langchain",
    "langchain_community",
]

print("📦 패키지 버전 확인")
print("=" * 40)

for pkg_name in packages:
    try:
        pkg = importlib.import_module(pkg_name)
        version = getattr(pkg, "__version__", "installed")
        print(f"  ✅ {pkg_name}: {version}")
    except ImportError:
        print(f"  ❌ {pkg_name}: 설치 필요")
        print(f"     pip install {pkg_name}")

## 2️⃣ 문서 로딩 (Document Loading)

RAG의 첫 단계는 **원본 문서를 로딩**하는 것입니다.

실습에서는 한국어 샘플 문서를 직접 생성하여 사용합니다.

In [ ]:
# 📄 실습용 한국어 샘플 문서 생성
sample_documents = [
    {
        "title": "인공지능의 역사",
        "content": """인공지능(AI)은 1956년 다트머스 회의에서 처음 학문 분야로 정립되었습니다. 
초기에는 규칙 기반 전문가 시스템이 주류였으나, 2012년 AlexNet의 등장으로 딥러닝 시대가 열렸습니다.
2017년 구글이 발표한 트랜스포머 아키텍처는 자연어 처리의 혁명을 가져왔습니다.
2022년 ChatGPT의 출시로 대규모 언어 모델(LLM)이 일반 대중에게 널리 알려지게 되었습니다.
현재 AI는 의료, 교육, 금융 등 거의 모든 산업 분야에서 활용되고 있습니다."""
    },
    {
        "title": "트랜스포머 아키텍처",
        "content": """트랜스포머는 2017년 'Attention is All You Need' 논문에서 소개된 딥러닝 모델 구조입니다.
셀프 어텐션(Self-Attention) 메커니즘을 사용하여 입력 시퀀스의 모든 위치를 동시에 참조합니다.
인코더-디코더 구조로 이루어져 있으며, BERT는 인코더만, GPT는 디코더만 사용합니다.
멀티헤드 어텐션은 서로 다른 관점에서 입력을 분석할 수 있게 해줍니다.
포지셔널 인코딩으로 순서 정보를 제공하며, 레이어 정규화와 잔차 연결로 학습을 안정화합니다."""
    },
    {
        "title": "파인튜닝 기법",
        "content": """파인튜닝은 사전학습된 모델을 특정 작업이나 도메인에 맞게 추가 학습하는 과정입니다.
Full Fine-Tuning(FFT)은 모든 파라미터를 업데이트하지만 많은 GPU 메모리가 필요합니다.
LoRA(Low-Rank Adaptation)는 작은 어댑터만 학습하여 메모리를 크게 절약합니다.
QLoRA는 4bit 양자화와 LoRA를 결합한 방법으로, 소비자급 GPU에서도 대형 모델 학습이 가능합니다.
SFT(Supervised Fine-Tuning)는 지시-응답 데이터셋으로 모델의 지시 따르기 능력을 향상시킵니다.
RLHF(Reinforcement Learning from Human Feedback)는 인간 선호도를 반영한 강화학습 방법입니다."""
    },
    {
        "title": "RAG 시스템 구축",
        "content": """RAG(Retrieval-Augmented Generation)는 검색 증강 생성이라고 합니다.
외부 지식 베이스에서 관련 정보를 검색한 후 LLM에 제공하여 답변을 생성합니다.
벡터 데이터베이스에 문서를 임베딩으로 저장하고, 질문과 유사한 문서를 검색합니다.
ChromaDB, Pinecone, Weaviate 등의 벡터 DB를 사용할 수 있습니다.
청킹 전략, 임베딩 모델 선택, 검색 알고리즘이 RAG 성능에 큰 영향을 미칩니다."""
    },
    {
        "title": "프롬프트 엔지니어링",
        "content": """프롬프트 엔지니어링은 LLM에서 원하는 응답을 이끌어내기 위한 입력 설계 기술입니다.
제로샷(Zero-shot) 프롬프팅은 예시 없이 직접 질문하는 방법입니다.
퓨샷(Few-shot) 프롬프팅은 몇 가지 예시를 함께 제공하여 성능을 높입니다.
Chain-of-Thought(CoT)는 단계별 추론 과정을 유도하여 복잡한 문제를 해결합니다.
시스템 프롬프트로 모델의 역할과 행동 규칙을 정의할 수 있습니다."""
    },
    {
        "title": "양자화 기술",
        "content": """양자화(Quantization)는 모델의 가중치를 낮은 정밀도로 변환하여 메모리와 연산을 줄이는 기술입니다.
FP32에서 FP16으로의 변환은 거의 품질 손실 없이 메모리를 절반으로 줄입니다.
INT8 양자화는 메모리를 75% 줄이면서 대부분의 성능을 유지합니다.
GPTQ와 AWQ는 학습 후 양자화(Post-Training Quantization) 기법입니다.
GGUF 포맷은 CPU에서도 효율적으로 추론할 수 있도록 설계되었습니다.
양자화는 모델 배포 시 하드웨어 요구사항을 크게 낮추어 줍니다."""
    },
]

print(f"📄 샘플 문서 {len(sample_documents)}개 준비 완료")
for i, doc in enumerate(sample_documents, 1):
    char_count = len(doc['content'])
    print(f"  {i}. {doc['title']} ({char_count}자)")

## 3️⃣ 문서 청킹 (Text Chunking)

긴 문서를 작은 조각(chunk)으로 나누는 과정입니다.

### 🔑 청킹이 중요한 이유
- 🔹 임베딩 모델의 입력 길이 제한
- 🔹 관련 없는 내용이 섞이면 검색 품질 저하
- 🔹 적절한 크기가 LLM 컨텍스트에 맞아야 함

### 📏 청킹 전략
| 전략 | 특징 | 적합한 경우 |
|------|------|------------|
| 고정 크기 | 일정 글자/토큰 수로 분할 | 일반적인 경우 |
| 문장 기반 | 문장 단위로 분할 | 짧은 문서 |
| 의미 기반 | 주제 변경 지점에서 분할 | 긴 문서 |
| 오버랩 | 청크 간 중복 구간 포함 | 문맥 유지 필요 시 |

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ✂️ 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,          # 각 청크의 최대 문자 수
    chunk_overlap=50,        # 청크 간 중복 문자 수
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],  # 우선순위 분할자
)

print("✂️ 문서 청킹 시작")
print("=" * 50)
print(f"📏 설정: chunk_size={200}, chunk_overlap={50}")
print()

all_chunks = []
all_metadatas = []

for doc in sample_documents:
    chunks = text_splitter.split_text(doc["content"])
    print(f"📄 '{doc['title']}' → {len(chunks)}개 청크")
    
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadatas.append({
            "title": doc["title"],
            "chunk_index": i,
            "source": f"{doc['title']}_chunk_{i}"
        })

print(f"\n📊 총 청크 수: {len(all_chunks)}개")

In [ ]:
# 🔍 청킹 결과 살펴보기
print("🔍 청킹 결과 예시 (첫 5개)")
print("=" * 60)

for i, (chunk, meta) in enumerate(zip(all_chunks[:5], all_metadatas[:5])):
    print(f"\n📌 청크 {i+1} (출처: {meta['title']}, 인덱스: {meta['chunk_index']})")
    print(f"   길이: {len(chunk)}자")
    print(f"   내용: {chunk[:100]}...")
    print("-" * 60)

## 4️⃣ 임베딩 생성 (Embedding)

텍스트를 **벡터(숫자 배열)**로 변환하여 의미적 유사도를 계산할 수 있게 합니다.

### 📐 임베딩이란?
- 텍스트 → 고차원 벡터 (예: 384차원)
- 의미가 비슷한 텍스트는 벡터 공간에서 가까움
- 코사인 유사도로 문서 간 관련성을 측정

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 📐 임베딩 모델 로딩
# 한국어 지원 임베딩 모델 사용
# 한국어 RAG 최적화 모델 (KLUE-RoBERTa + SimCSE 대조학습)
# 다른 옵션:
#   "sentence-transformers/all-MiniLM-L6-v2"           — 영어 위주, 가벼움 (384d)
#   "paraphrase-multilingual-mpnet-base-v2"            — 다국어 균형 (768d)
#   "BM-K/KoSimCSE-bert-multitask"                     — KoSimCSE BERT 변형
EMBEDDING_MODEL = "BM-K/KoSimCSE-roberta-multitask"

print(f"📐 임베딩 모델 로딩: {EMBEDDING_MODEL}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"✅ 임베딩 모델 로딩 완료!")
print(f"📊 임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")

In [ ]:
# 🔢 임베딩 동작 이해하기
print("🔢 임베딩 동작 이해하기")
print("=" * 50)

test_sentences = [
    "인공지능은 미래를 바꿀 기술입니다.",
    "AI가 세상을 변화시키고 있습니다.",
    "오늘 점심에 김치찌개를 먹었습니다.",
]

# 임베딩 생성
embeddings = embedding_model.encode(test_sentences)

print(f"입력: {len(test_sentences)}개 문장")
print(f"출력: {embeddings.shape} (문장 수 x 차원)")
print(f"\n각 임베딩 벡터의 처음 5개 값:")
for i, (sent, emb) in enumerate(zip(test_sentences, embeddings)):
    print(f"  {i+1}. '{sent[:25]}...'")
    print(f"     [{emb[0]:.4f}, {emb[1]:.4f}, {emb[2]:.4f}, {emb[3]:.4f}, {emb[4]:.4f}, ...]")

# 코사인 유사도 계산
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print(f"\n📊 코사인 유사도:")
sim_01 = cosine_similarity(embeddings[0], embeddings[1])
sim_02 = cosine_similarity(embeddings[0], embeddings[2])
sim_12 = cosine_similarity(embeddings[1], embeddings[2])

print(f"  문장1 ↔ 문장2 (유사): {sim_01:.4f}")
print(f"  문장1 ↔ 문장3 (다름): {sim_02:.4f}")
print(f"  문장2 ↔ 문장3 (다름): {sim_12:.4f}")
print(f"\n💡 의미가 비슷한 문장1과 문장2의 유사도가 더 높습니다!")

## 5️⃣ ChromaDB에 벡터 저장

**ChromaDB**는 오픈소스 벡터 데이터베이스로, 임베딩을 저장하고 검색하는 데 사용합니다.

### 📊 주요 벡터 DB 비교
| DB | 특징 | 가격 |
|-----|------|------|
| **ChromaDB** | 오픈소스, 간단한 API | 무료 |
| Pinecone | 클라우드, 관리형 | 유/무료 |
| Weaviate | 오픈소스, GraphQL | 무료 |
| FAISS | Meta, 초고속 | 무료 |

In [ ]:
import chromadb

# 🗃️ ChromaDB 클라이언트 생성
# 메모리 내(in-memory) 또는 디스크 기반으로 사용 가능
chroma_client = chromadb.Client()  # 메모리 내 (실습용)
# chroma_client = chromadb.PersistentClient(path="./chroma_db")  # 디스크 저장

print("🗃️ ChromaDB 클라이언트 생성 완료")

# 컬렉션 생성
collection_name = "ai_knowledge_base"

# 기존 컬렉션이 있으면 삭제
try:
    chroma_client.delete_collection(collection_name)
except:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}  # 코사인 유사도 사용
)

print(f"✅ 컬렉션 '{collection_name}' 생성 완료")

In [ ]:
# 📥 청크를 ChromaDB에 저장
print("📥 문서 청크를 ChromaDB에 저장 중...")

# 임베딩 생성
print(f"  🔢 {len(all_chunks)}개 청크 임베딩 생성 중...")
chunk_embeddings = embedding_model.encode(all_chunks).tolist()
print(f"  ✅ 임베딩 생성 완료")

# ChromaDB에 추가
ids = [f"chunk_{i}" for i in range(len(all_chunks))]

collection.add(
    ids=ids,
    embeddings=chunk_embeddings,
    documents=all_chunks,
    metadatas=all_metadatas,
)

print(f"\n✅ ChromaDB 저장 완료!")
print(f"📊 저장된 문서 수: {collection.count()}")

## 6️⃣ 시맨틱 서치 (Semantic Search)

키워드 매칭이 아닌, **의미 기반**으로 관련 문서를 검색합니다.

In [ ]:
def semantic_search(query, n_results=3):
    """시맨틱 서치를 수행합니다."""
    # 쿼리 임베딩 생성
    query_embedding = embedding_model.encode([query]).tolist()
    
    # ChromaDB에서 검색
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    
    return results

# 🔍 검색 테스트 1
query = "LoRA와 QLoRA의 차이점은 무엇인가요?"
print(f"🔍 검색 쿼리: '{query}'")
print("=" * 60)

results = semantic_search(query, n_results=3)

for i in range(len(results['documents'][0])):
    doc = results['documents'][0][i]
    meta = results['metadatas'][0][i]
    distance = results['distances'][0][i]
    similarity = 1 - distance  # 코사인 거리 → 유사도
    
    print(f"\n📌 결과 {i+1} (유사도: {similarity:.4f})")
    print(f"   출처: {meta['title']}")
    print(f"   내용: {doc[:150]}...")

In [ ]:
# 🔍 다양한 쿼리로 검색 테스트
test_queries = [
    "트랜스포머 모델의 어텐션 메커니즘이 뭐야?",
    "모델을 작게 만드는 방법은?",
    "ChatGPT는 언제 나왔나요?",
    "벡터 데이터베이스 종류",
    "프롬프트를 잘 작성하는 방법",
]

print("🔍 다양한 쿼리로 시맨틱 서치 테스트")
print("=" * 70)

for query in test_queries:
    results = semantic_search(query, n_results=2)
    
    print(f"\n❓ 쿼리: '{query}'")
    for i in range(len(results['documents'][0])):
        meta = results['metadatas'][0][i]
        distance = results['distances'][0][i]
        similarity = 1 - distance
        print(f"   {i+1}. [{meta['title']}] 유사도: {similarity:.4f}")
    print("-" * 70)

## 7️⃣ 키워드 검색 vs 시맨틱 서치 비교

전통적인 키워드 매칭과 시맨틱 서치의 차이를 확인합니다.

In [ ]:
# 🔑 키워드 검색 함수
def keyword_search(query, documents, n_results=3):
    """단순 키워드 매칭 기반 검색"""
    query_words = set(query.lower().split())
    scores = []
    
    for i, doc in enumerate(documents):
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)
        scores.append((i, overlap, doc))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:n_results]

# 🔍 비교 테스트
compare_queries = [
    "모델을 가볍게 만드는 방법",  # '양자화'라는 단어 없이 검색
    "AI의 발전 역사",            # 직접적인 키워드가 없는 검색
]

print("📊 키워드 검색 vs 시맨틱 서치 비교")
print("=" * 70)

for query in compare_queries:
    print(f"\n❓ 쿼리: '{query}'")
    
    # 키워드 검색
    keyword_results = keyword_search(query, all_chunks, n_results=2)
    print(f"\n  🔑 키워드 검색 결과:")
    for idx, score, doc in keyword_results:
        title = all_metadatas[idx]['title']
        print(f"    [{title}] 매칭 단어: {score}개 | {doc[:60]}...")
    
    # 시맨틱 서치
    semantic_results = semantic_search(query, n_results=2)
    print(f"\n  🧠 시맨틱 서치 결과:")
    for i in range(len(semantic_results['documents'][0])):
        meta = semantic_results['metadatas'][0][i]
        doc = semantic_results['documents'][0][i]
        distance = semantic_results['distances'][0][i]
        similarity = 1 - distance
        print(f"    [{meta['title']}] 유사도: {similarity:.4f} | {doc[:60]}...")
    
    print("-" * 70)

## 8️⃣ 청킹 전략 실험

chunk_size와 chunk_overlap을 변경하면 검색 결과가 어떻게 달라지는지 확인합니다.

In [ ]:
# 📏 다양한 청킹 설정 비교
chunking_configs = [
    {"chunk_size": 100, "chunk_overlap": 20, "label": "작은 청크 (100자)"},
    {"chunk_size": 200, "chunk_overlap": 50, "label": "중간 청크 (200자)"},
    {"chunk_size": 500, "chunk_overlap": 100, "label": "큰 청크 (500자)"},
]

print("📏 청킹 전략별 결과 비교")
print("=" * 60)

# 하나의 긴 문서로 테스트
test_text = sample_documents[2]["content"]  # 파인튜닝 기법
print(f"원본 문서: '{sample_documents[2]['title']}' ({len(test_text)}자)")

for config in chunking_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
    )
    chunks = splitter.split_text(test_text)
    
    avg_len = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
    
    print(f"\n🔹 {config['label']} (overlap: {config['chunk_overlap']})")
    print(f"   청크 수: {len(chunks)}개")
    print(f"   평균 길이: {avg_len:.0f}자")
    for i, chunk in enumerate(chunks[:3]):
        print(f"   [{i+1}] {chunk[:60]}...")

In [ ]:
# 📌 실습 정리
print("📌 오늘의 핵심 정리")
print("=" * 50)
print("  1️⃣ RAG는 외부 지식을 검색하여 LLM 응답을 강화하는 기술")
print("  2️⃣ 문서를 적절한 크기로 청킹하는 것이 중요")
print("  3️⃣ 임베딩은 텍스트를 벡터로 변환하여 의미 비교 가능")
print("  4️⃣ ChromaDB로 벡터를 저장하고 시맨틱 서치 수행")
print("  5️⃣ 시맨틱 서치는 키워드 매칭보다 의미적으로 정확")
print("  6️⃣ 청킹 전략(크기, 오버랩)이 검색 품질에 영향")

---

## 🎯 실습 과제

1️⃣ 자신만의 문서 3개를 추가하고 ChromaDB에 저장해보세요  
2️⃣ chunk_size를 150, 300, 500으로 바꿔가며 검색 결과를 비교해보세요  
3️⃣ 다른 임베딩 모델(`intfloat/multilingual-e5-small`)로 교체하고 결과를 비교해보세요  

---

## 📚 참고 자료
- [ChromaDB 공식 문서](https://docs.trychroma.com/)
- [Sentence Transformers](https://www.sbert.net/)
- [LangChain Text Splitters](https://python.langchain.com/docs/modules/data_connection/document_transformers/)